# Welly AI RAG Notebook

Notebook นี้ถูกปรับให้ใช้แนวทาง **strict / dataset-grounded RAG** สำหรับโปรเจกต์ Welly AI

แนวคิดหลักมี 2 ชั้น:
- ใช้ embeddings + FAISS สำหรับ retrieval diagnostics
- ใช้ strict answer engine จาก `Src/model.py` สำหรับคำตอบสุดท้าย เพื่อหลีกเลี่ยง hallucination

หลักการที่ notebook นี้ยึด:
- ตอบจากข้อมูลใน dataset เท่านั้น
- ถ้าข้อมูลไม่พอ หรือชื่ออาหารไม่ชัด ต้องยอมบอกว่าไม่พอ
- ถ้าคำถามอยู่นอกขอบเขตของ dataset ต้องปฏิเสธ
- ไม่ใช้ `full_knowledge_base.csv` ซ้ำกับ raw tables ใน answer path


In [2]:
# ติดตั้งแพ็กเกจที่จำเป็น (รันใน Colab หรือ notebook environment ที่ยังไม่มี library)
!pip -q install -U langchain langchain-community langchain-core langchain-text-splitters \
    langchain-huggingface langchain-groq faiss-cpu sentence-transformers rapidfuzz python-Levenshtein


In [3]:
%pip install -U langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface langchain-groq faiss-cpu sentence-transformers rapidfuzz python-Levenshtein

  Using cached langchain_core-1.2.24-py3-none-any.whl.metadata (4.4 kB)
Using cached langchain_core-1.2.24-py3-none-any.whl (506 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.23
    Uninstalling langchain-core-1.2.23:
      Successfully uninstalled langchain-core-1.2.23

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import sys
print(sys.executable)

/usr/local/bin/python3


In [5]:
import rapidfuzz
import langchain_core
import langchain_huggingface
import langchain_community
import langchain_text_splitters
import langchain_groq

print("all imports ok")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


all imports ok


## 1. Import Libraries

In [6]:
import os
import json
import re
from pathlib import Path

import pandas as pd

from rapidfuzz import process
from difflib import get_close_matches

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA


## 2. Load Secrets

ใช้ `LC_TOKEN` และ `GROQ_TOKEN` จาก Colab Secrets หรือ environment variables  
ถ้าไม่มี token จะยังสร้าง vector store และทดสอบ retrieval ได้ แต่จะยังใช้ LLM ตอบเต็มรูปไม่ได้


In [7]:
from getpass import getpass

LC_TOKEN = os.environ.get("LC_TOKEN")
GROQ_TOKEN = os.environ.get("GROQ_TOKEN")

# Fallback: ask user to input token in notebook if env var is missing.
if not GROQ_TOKEN:
    try:
        GROQ_TOKEN = getpass("Enter GROQ_TOKEN: ").strip()
    except Exception:
        GROQ_TOKEN = input("Enter GROQ_TOKEN: ").strip()

if LC_TOKEN:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_API_KEY"] = LC_TOKEN

if GROQ_TOKEN:
    os.environ["GROQ_API_KEY"] = GROQ_TOKEN

print("LC_TOKEN loaded from environment:", LC_TOKEN is not None)
print("GROQ_TOKEN ready:", bool(GROQ_TOKEN))


LC_TOKEN loaded from environment: False
GROQ_TOKEN ready: True


## 3. Locate Project Files

In [8]:
knowledge_dir = Path("../data/knowledge")
outputs_dir = Path("../outputs")

candidate_files = [
    knowledge_dir / "standard_df.csv",
    knowledge_dir / "dga_standard_df.csv",
    knowledge_dir / "dga_rules_df.csv",
    knowledge_dir / "bmi_standard_df.csv",
    knowledge_dir / "bmi_rules_df.csv",
    knowledge_dir / "claim_rules_df.csv",
    knowledge_dir / "label_required_nutrients_df.csv",
    knowledge_dir / "serving_size_reference_df.csv",
    knowledge_dir / "user_health_knowledge.csv",
    outputs_dir / "food_dataset_with_risk.csv",
]

file_status = pd.DataFrame({
    "file": [str(p) for p in candidate_files],
    "exists": [p.exists() for p in candidate_files]
})
file_status


,file,exists
0,../data/knowledge/standard_df.csv,True
1,../data/knowledge/dga_standard_df.csv,True
2,../data/knowledge/dga_rules_df.csv,True
3,../data/knowledge/bmi_standard_df.csv,True
4,../data/knowledge/bmi_rules_df.csv,True
5,../data/knowledge/claim_rules_df.csv,True
6,../data/knowledge/label_required_nutrients_df.csv,True
7,../data/knowledge/serving_size_reference_df.csv,True
8,../data/knowledge/user_health_knowledge.csv,True
9,../outputs/food_dataset_with_risk.csv,True


## 4. Load Tables

In [9]:
loaded_tables = {}

for path in candidate_files:
    if path.exists():
        try:
            loaded_tables[path.name] = pd.read_csv(path)
        except Exception as e:
            print(f"Failed to load {path.name}: {e}")

print("Loaded tables:", list(loaded_tables.keys()))
for name, df in loaded_tables.items():
    print(name, df.shape)


Loaded tables: ['standard_df.csv', 'dga_standard_df.csv', 'dga_rules_df.csv', 'bmi_standard_df.csv', 'bmi_rules_df.csv', 'claim_rules_df.csv', 'label_required_nutrients_df.csv', 'serving_size_reference_df.csv', 'user_health_knowledge.csv', 'food_dataset_with_risk.csv']
standard_df.csv (7, 6)
dga_standard_df.csv (8, 7)
dga_rules_df.csv (4, 4)
bmi_standard_df.csv (10, 9)
bmi_rules_df.csv (3, 5)
claim_rules_df.csv (3, 6)
label_required_nutrients_df.csv (9, 4)
serving_size_reference_df.csv (7, 5)
user_health_knowledge.csv (5000, 17)
food_dataset_with_risk.csv (2395, 20)


## 5. Preview Tables

In [10]:
for name, df in loaded_tables.items():
    print("=" * 100)
    print(name)
    display(df.head(3))


standard_df.csv


,source_doc,category,metric,recommended_value,unit,note
0,media.pdf,daily_reference,Energy,2000,kcal/day,Thai RDI reference base
1,media.pdf,daily_reference,Total Fat,65,g/day,Thai RDI reference
2,media.pdf,daily_reference,Saturated Fat,20,g/day,Thai RDI reference


dga_standard_df.csv


,source_doc,category,metric,min_value,max_value,unit,target_group
0,DGA.pdf,Protein,protein_intake,1.2,1.6,g/kg/day,general
1,DGA.pdf,Dairy,dairy_servings,3.0,3.0,servings/day,2000_kcal_pattern
2,DGA.pdf,Vegetables,vegetable_servings,3.0,3.0,servings/day,2000_kcal_pattern


dga_rules_df.csv


...

,source_doc,category,item,reference_serving,unit
0,media.pdf,Dairy,Ready-to-drink milk,200,ml
1,media.pdf,Beverage,Ready-to-drink beverage,200,ml
2,media.pdf,Snack,Chips / popcorn / crispy snacks,30,g


user_health_knowledge.csv


,Patient_ID,Age,Gender,Height_cm,Weight_kg,BMI,BMI_Category,Blood_Pressure_Systolic,Blood_Pressure_Diastolic,Blood_Sugar_Level,Cholesterol_Level,Health_Profile_Summary,Recommended_Calories,Recommended_Protein,Recommended_Carbs,Recommended_Fats,Recommended_Meal_Plan
0,P00001,56,Other,163,66,24.84,Overweight,175,75,124,219,Overall moderate cardiometabolic risk. BMI: Ov...,2150,108,139,145,High-Protein Diet
1,P00002,69,Female,171,114,38.99,Obese Level 2,155,72,72,208,Overall moderate cardiometabolic risk. BMI: Ob...,1527,74,266,80,Balanced Diet
2,P00003,46,Female,172,119,40.22,Obese Level 2,137,101,145,171,Overall high cardiometabolic risk. BMI: Obese ...,2359,180,145,143,High-Protein Diet


food_dataset_with_risk.csv


,food_name,calories,fat,sat_fat,carbs,sugar,protein,fiber,cholesterol,sodium,sodium_pct_daily,cholesterol_pct_daily,sat_fat_pct_daily,fat_pct_daily,carbs_pct_daily,fiber_pct_daily,sugar_pct_meal_limit,risk_level,risk_label,chatbot_summary
0,cream cheese,51,5.0,2.9,0.8,0.5,0.9,0.0,14.6,0.016,0.0008,4.866667,14.5,7.692308,0.266667,0.0,5.0,Low,0,ยังไม่พบตัวชี้วัดที่เกินเกณฑ์เบื้องต้น
1,neufchatel cheese,215,19.4,10.9,3.1,2.7,7.8,0.0,62.9,0.300,0.0150,20.966667,54.5,29.846154,1.033333,0.0,27.0,Medium,1,โคเลสเตอรอลค่อนข้างสูง | ไขมันอิ่มตัวค่อนข้างสูง
2,requeijao cremoso light catupiry,49,3.6,2.3,0.9,3.4,0.8,0.1,0.0,0.000,0.0000,0.000000,11.5,5.538462,0.300000,0.4,34.0,Low,0,ยังไม่พบตัวชี้วัดที่เกินเกณฑ์เบื้องต้น


## 6. Convert Tables to Documents

จะแปลงแต่ละ row ให้เป็นข้อความสั้น ๆ พร้อม metadata  
เพื่อให้ retrieval ดึงข้อมูลได้ตรงขึ้น


In [11]:
def row_to_text(table_name, row):
    table = table_name.lower()

    if table == "standard_df.csv":
        return (
            f"Nutrition standard. Source {row.get('source_doc', '')}. "
            f"Category {row.get('category', '')}. Metric {row.get('metric', '')}. "
            f"Recommended value {row.get('recommended_value', '')} {row.get('unit', '')}. "
            f"Note {row.get('note', '')}."
        )

    if table == "dga_standard_df.csv":
        return (
            f"DGA standard. Source {row.get('source_doc', '')}. Category {row.get('category', '')}. "
            f"Metric {row.get('metric', '')}. Min value {row.get('min_value', '')}. "
            f"Max value {row.get('max_value', '')} {row.get('unit', '')}. "
            f"Target group {row.get('target_group', '')}."
        )

    if table == "dga_rules_df.csv":
        return (
            f"DGA rule. Source {row.get('source_doc', '')}. Rule name {row.get('rule_name', '')}. "
            f"Condition {row.get('condition', '')}. Unit {row.get('unit', '')}."
        )

    if table == "bmi_standard_df.csv":
        return (
            f"BMI standard. Source {row.get('source_doc', '')}. Category {row.get('category', '')}. "
            f"Metric {row.get('metric', '')}. Label {row.get('label', '')}. "
            f"Min value {row.get('min_value', '')}. Max value {row.get('max_value', '')}. "
            f"Unit {row.get('unit', '')}. Target group {row.get('target_group', '')}. "
            f"Note {row.get('note', '')}."
        )

    if table == "bmi_rules_df.csv":
        return (
            f"BMI rule. Source {row.get('source_doc', '')}. Rule name {row.get('rule_name', '')}. "
            f"Condition {row.get('condition', '')}. Unit {row.get('unit', '')}. "
            f"Note {row.get('note', '')}."
        )

    if table == "claim_rules_df.csv":
        return (
            f"Claim rule. Source {row.get('source_doc', '')}. Rule name {row.get('rule_name', '')}. "
            f"Condition type {row.get('condition_type', '')}. Nutrient {row.get('nutrient', '')}. "
            f"Value {row.get('value', '')} {row.get('unit', '')}."
        )

    if table == "label_required_nutrients_df.csv":
        return (
            f"Label required nutrient. Source {row.get('source_doc', '')}. Section {row.get('section', '')}. "
            f"Nutrient {row.get('nutrient', '')}. Unit {row.get('unit', '')}."
        )

    if table == "serving_size_reference_df.csv":
        return (
            f"Serving size reference. Source {row.get('source_doc', '')}. Category {row.get('category', '')}. "
            f"Item {row.get('item', '')}. Reference serving {row.get('reference_serving', '')} {row.get('unit', '')}."
        )

    if table == "user_health_knowledge.csv":
        return (
            f"User health knowledge. Patient ID {row.get('Patient_ID', row.get('patient_id', ''))}. "
            f"Age {row.get('Age', '')}. Gender {row.get('Gender', '')}. "
            f"Height {row.get('Height_cm', '')} cm. Weight {row.get('Weight_kg', '')} kg. "
            f"BMI {row.get('BMI', '')}. BMI category {row.get('BMI_Category', '')}. "
            f"Blood pressure systolic {row.get('Blood_Pressure_Systolic', '')}. "
            f"Blood pressure diastolic {row.get('Blood_Pressure_Diastolic', '')}. "
            f"Blood sugar {row.get('Blood_Sugar_Level', '')}. "
            f"Cholesterol {row.get('Cholesterol_Level', '')}. "
            f"Recommended calories {row.get('Recommended_Calories', '')}. "
            f"Recommended protein {row.get('Recommended_Protein', '')}. "
            f"Recommended carbs {row.get('Recommended_Carbs', '')}. "
            f"Recommended fats {row.get('Recommended_Fats', '')}. "
            f"Recommended meal plan {row.get('Recommended_Meal_Plan', '')}. "
            f"Health summary {row.get('Health_Profile_Summary', '')}."
        )

    if table == "food_dataset_with_risk.csv":
        return (
            f"Food risk record. Food name {row.get('food_name', '')}. "
            f"Calories {row.get('calories', '')}. Fat {row.get('fat', '')}. "
            f"Saturated fat {row.get('sat_fat', '')}. Carbs {row.get('carbs', '')}. "
            f"Sugar {row.get('sugar', '')}. Protein {row.get('protein', '')}. "
            f"Fiber {row.get('fiber', '')}. Cholesterol {row.get('cholesterol', '')}. "
            f"Sodium {row.get('sodium', '')}. Risk level {row.get('risk_level', '')}. "
            f"Chatbot summary {row.get('chatbot_summary', '')}."
        )

    joined = ". ".join([f"{col}: {row.get(col, '')}" for col in row.index])
    return f"{table_name} record. {joined}"


In [12]:
documents = []

for table_name, df in loaded_tables.items():
    for idx, row in df.iterrows():
        text = row_to_text(table_name, row)
        doc = Document(
            page_content=text,
            metadata={
                "table": table_name,
                "row_index": int(idx)
            }
        )
        documents.append(doc)

print("Total documents:", len(documents))
documents[:3]


Total documents: 7446


[Document(metadata={'table': 'standard_df.csv', 'row_index': 0}, page_content='Nutrition standard. Source media.pdf. Category daily_reference. Metric Energy. Recommended value 2000 kcal/day. Note Thai RDI reference base.'),
 Document(metadata={'table': 'standard_df.csv', 'row_index': 1}, page_content='Nutrition standard. Source media.pdf. Category daily_reference. Metric Total Fat. Recommended value 65 g/day. Note Thai RDI reference.'),
 Document(metadata={'table': 'standard_df.csv', 'row_index': 2}, page_content='Nutrition standard. Source media.pdf. Category daily_reference. Metric Saturated Fat. Recommended value 20 g/day. Note Thai RDI reference.')]

## 7. Optional Text Splitting

ถ้าเอกสารยาวมากสามารถ split ได้  
แต่กรณีนี้แต่ละ row สั้นอยู่แล้ว จึง split แบบเบา ๆ หรือข้ามได้


In [13]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

split_docs = splitter.split_documents(documents)
print("Split documents:", len(split_docs))
split_docs[:2]


Split documents: 12446


[Document(metadata={'table': 'standard_df.csv', 'row_index': 0}, page_content='Nutrition standard. Source media.pdf. Category daily_reference. Metric Energy. Recommended value 2000 kcal/day. Note Thai RDI reference base.'),
 Document(metadata={'table': 'standard_df.csv', 'row_index': 1}, page_content='Nutrition standard. Source media.pdf. Category daily_reference. Metric Total Fat. Recommended value 65 g/day. Note Thai RDI reference.')]

## 8. Build Embeddings + Vector Store

In [14]:
# Use a smaller multilingual model for much faster embedding on CPU.
model_name = "intfloat/multilingual-e5-small"
model_tag = model_name.split("/")[-1].replace("-", "_")

try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    device = "cpu"

print("Embedding device:", device)
print("Embedding model:", model_name)

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 64},
)

index_dir = Path(f"faiss_welly_index_{model_tag}")
index_faiss = index_dir / "index.faiss"
index_pkl = index_dir / "index.pkl"

if index_faiss.exists() and index_pkl.exists():
    vectorstore = FAISS.load_local(str(index_dir), embeddings, allow_dangerous_deserialization=True)
    print("Loaded existing knowledge FAISS index")
else:
    vectorstore = FAISS.from_documents(split_docs, embeddings)
    index_dir.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(index_dir))
    print("Built and saved knowledge FAISS index")


Embedding device: cpu
Embedding model: intfloat/multilingual-e5-small


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16960.97it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded existing knowledge FAISS index


## 9. Quick Retrieval Test

In [15]:
test_queries = [
    "BMI ปกติคือเท่าไร",
    "โซเดียมต่อวันควรไม่เกินเท่าไร",
    "น้ำตาลต่อมื้อควรไม่เกินเท่าไร",
    "cream cheese เสี่ยงไหม",
    "อาหารที่มีคอเลสเตอรอลสูง"
]

for q in test_queries:
    print("=" * 100)
    print("Query:", q)
    results = vectorstore.similarity_search(q, k=3)
    for i, doc in enumerate(results, start=1):
        print(f"{i}. [{doc.metadata.get('table')}] {doc.page_content[:300]}")
    print()


Query: BMI ปกติคือเท่าไร
1. [bmi_standard_df.csv] BMI standard. Source document-20210831192536.pdf. Category BMI. Metric bmi_category. Label Normal. Min value 18.5. Max value 22.99. Unit kg/m^2. Target group asian_adults. Note น้ำหนักปกติ.
2. [bmi_standard_df.csv] BMI standard. Source document-20210831192536.pdf. Category BMI. Metric bmi_category. Label Obese Level 1. Min value 25.0. Max value 29.99. Unit kg/m^2. Target group asian_adults. Note อ้วนระดับ 1.
3. [bmi_standard_df.csv] BMI standard. Source document-20210831192536.pdf. Category BMI. Metric bmi_category. Label Overweight. Min value 23.0. Max value 24.99. Unit kg/m^2. Target group asian_adults. Note น้ำหนักเกิน.

Query: โซเดียมต่อวันควรไม่เกินเท่าไร
1. [food_dataset_with_risk.csv] Food risk record. Food name baking soda. Calories 0. Fat 0.0. Saturated fat 0.0. Carbs 0.0. Sugar 0.0. Protein 0.0. Fiber 0.0. Cholesterol 0.0. Sodium 1.3. Risk level Low. Chatbot summary ยังไม่พบตัวชี้วัดที่เกินเกณฑ์เบื้องต้น.
2. [food_dataset_with

## 10. Setup Retriever

In [16]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")


Retriever ready


## 11. Setup LLM Answer Engine

In [17]:
if not GROQ_TOKEN:
    raise ValueError("GROQ_TOKEN not found. Please set environment variable before running LLM QA.")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_tokens=512,
)

rag_prompt = PromptTemplate.from_template(
    """
You are Welly AI nutrition assistant for human nutrition.
Answer in Thai with a natural, polite chat tone.
Use ONLY the provided context and never use outside knowledge.

When context is sufficient:
- Answer directly and clearly in normal chat style.
- Keep it concise and practical.

When context is insufficient:
- Reply exactly:
"ตอนนี้ผมยังไม่มีข้อมูลที่เพียงพอเกี่ยวกับ '{question}'\nหากคุณต้องการ ผมช่วยแนะนำคำถามด้านโภชนาการหรือสูตรอาหารที่ใกล้เคียงให้ได้ครับ"

When out of scope (pet/animal food):
- Reply exactly:
"หัวข้อนี้อยู่นอกขอบเขตของผู้ช่วยโภชนาการสำหรับมนุษย์ครับ\nผมช่วยตอบเรื่องโภชนาการ อาหาร และสุขภาพของคนได้"

Question:
{question}

Context:
{context}

Answer:
""".strip()
)

print("LLM answer engine ready")


LLM answer engine ready


## 12. Ask Questions

In [18]:
def _search_scored(vs, query: str, k: int = 4):
    try:
        return vs.similarity_search_with_relevance_scores(query, k=k)
    except AssertionError as e:
        print(f"Warning: vector search skipped due to index mismatch: {e}")
        return []
    except Exception:
        try:
            docs = vs.similarity_search(query, k=k)
            return [(d, 0.0) for d in docs]
        except Exception as e:
            print(f"Warning: vector search failed: {e}")
            return []


def _detect_intent(question: str) -> str:
    q = question.lower()
    calorie_keywords = ["แคล", "แคลอ", "kcal", "calorie", "calories", "พลังงาน"]
    recipe_keywords = ["วิธีทำ", "ทำยังไง", "ขั้นตอน", "สูตร", "เมนู", "recipe", "cook", "ingredients"]
    guideline_keywords = ["ควร", "ไม่เกิน", "ปกติ", "มาตรฐาน", "ต่อวัน", "ต่อมื้อ", "bmi", "โซเดียม", "น้ำตาล"]

    if any(k in q for k in calorie_keywords):
        return "calorie"
    if any(k in q for k in recipe_keywords):
        return "recipe"
    if any(k in q for k in guideline_keywords):
        return "guideline"
    return "general"


def _is_out_of_scope(question: str) -> bool:
    q = question.lower()
    out_scope_keywords = [
        "อาหารหมา", "อาหารแมว", "อาหารสุนัข", "อาหารสัตว์", "หมากิน", "แมวกิน",
        "dog food", "cat food", "pet food", "animal feed"
    ]
    return any(k in q for k in out_scope_keywords)


def _default_fallback(question: str) -> str:
    return (
        f"ตอนนี้ผมยังไม่มีข้อมูลที่เพียงพอเกี่ยวกับ '{question}'\n"
        "หากคุณต้องการ ผมช่วยแนะนำคำถามด้านโภชนาการหรือสูตรอาหารที่ใกล้เคียงให้ได้ครับ"
    )


def _clean_chat_answer(text: str, intent: str, question: str) -> str:
    ans = (text or "").strip()

    if not ans:
        return _default_fallback(question)

    # Keep strict fallback/out-of-scope messages as-is.
    if intent == "out_of_scope" or "ตอนนี้ผมยังไม่มีข้อมูลที่เพียงพอ" in ans or "อยู่นอกขอบเขต" in ans:
        return ans

    # Remove accidental JSON wrappers if model returns structured text.
    if ans.startswith("{") and ans.endswith("}"):
        try:
            obj = json.loads(ans)
            if isinstance(obj, dict) and obj.get("answer"):
                ans = str(obj.get("answer")).strip()
        except Exception:
            pass

    # Make tone more direct and less repetitive apology.
    ans = re.sub(r"^(ขออภัยด้วยครับ|ขอโทษครับ|ขออภัยครับ)\s*", "", ans)
    ans = ans.strip()

    return ans if ans else _default_fallback(question)


def _to_float(value):
    if value is None:
        return None
    s = str(value).strip()
    if not s:
        return None
    s = re.sub(r"[^0-9.\-]", "", s)
    if not s:
        return None
    try:
        return float(s)
    except Exception:
        return None


def _answer_high_cholesterol_examples(question: str):
    q = question.lower()
    cholesterol_keywords = ["คอเลสเตอรอล", "cholesterol"]
    example_keywords = ["ตัวอย่าง", "มีอะไรบ้าง", "อะไรบ้าง", "high", "สูง"]
    if not any(k in q for k in cholesterol_keywords):
        return None
    if not any(k in q for k in example_keywords):
        return None

    df = loaded_tables.get("food_dataset_with_risk.csv")
    if df is None or df.empty:
        return None
    if "food_name" not in df.columns or "cholesterol" not in df.columns:
        return None

    work = df[["food_name", "cholesterol"]].copy()
    work["cholesterol_num"] = work["cholesterol"].apply(_to_float)
    work = work.dropna(subset=["food_name", "cholesterol_num"])
    if work.empty:
        return None

    work = work[work["cholesterol_num"] > 0].sort_values("cholesterol_num", ascending=False)
    top = work.head(5)
    if top.empty:
        return None

    lines = [
        "ตัวอย่างอาหารที่มีคอเลสเตอรอลสูง (จาก dataset):"
    ]
    for _, row in top.iterrows():
        lines.append(f"- {str(row['food_name']).strip()}: {row['cholesterol_num']:.1f}")

    answer = "\n".join(lines)
    sources = [{"table": "food_dataset_with_risk.csv", "retrieved_from": "knowledge", "method": "tabular_top_cholesterol"}]
    return {
        "question": question,
        "intent": "cholesterol_examples",
        "answer": answer,
        "sources": sources,
    }


def _answer_daily_sugar_limit(question: str):
    q = question.lower()
    if not (("น้ำตาล" in q or "sugar" in q) and ("ต่อวัน" in q or "ไม่เกิน" in q or "ควร" in q)):
        return None

    candidate_tables = ["dga_rules_df.csv", "dga_standard_df.csv", "standard_df.csv"]
    snippets = []
    used_sources = []

    for tname in candidate_tables:
        df = loaded_tables.get(tname)
        if df is None or df.empty:
            continue

        mask = pd.Series(False, index=df.index)
        for col in df.columns:
            col_text = str(col).lower()
            if any(key in col_text for key in ["sugar", "น้ำตาล", "rule", "metric", "condition", "note"]):
                mask = mask | df[col].astype(str).str.contains("sugar|น้ำตาล", case=False, regex=True, na=False)

        sub = df[mask].head(3)
        if sub.empty:
            continue

        used_sources.append({"table": tname, "retrieved_from": "knowledge", "method": "tabular_sugar_lookup"})
        for _, row in sub.iterrows():
            row_text = ". ".join([f"{c}: {row[c]}" for c in row.index if str(row[c]).strip() != ""])
            snippets.append(row_text)

    if not snippets:
        return None

    context = "\n".join(snippets[:4])
    prompt_text = rag_prompt.format(question=question, context=context)
    llm_response = llm.invoke(prompt_text)
    raw_answer = llm_response.content if hasattr(llm_response, "content") else str(llm_response)
    final_answer = _clean_chat_answer(raw_answer, intent="guideline", question=question)

    return {
        "question": question,
        "intent": "guideline",
        "answer": final_answer,
        "sources": used_sources,
    }


def ask_welly_rag(question: str, k: int = 4):
    if _is_out_of_scope(question):
        return {
            "question": question,
            "intent": "out_of_scope",
            "answer": "หัวข้อนี้อยู่นอกขอบเขตของผู้ช่วยโภชนาการสำหรับมนุษย์ครับ\nผมช่วยตอบเรื่องโภชนาการ อาหาร และสุขภาพของคนได้",
            "sources": [],
        }

    cholesterol_direct = _answer_high_cholesterol_examples(question)
    if cholesterol_direct is not None:
        return cholesterol_direct

    sugar_direct = _answer_daily_sugar_limit(question)
    if sugar_direct is not None:
        return sugar_direct

    intent = _detect_intent(question)

    store_plan = [("knowledge", vectorstore)]
    if intent == "calorie":
        store_plan.insert(0, ("calorie", calories_vectorstore))
    elif intent == "recipe":
        store_plan.insert(0, ("recipe", recipes_vectorstore))

    collected = []
    for source_name, vs in store_plan:
        hits = _search_scored(vs, question, k=k)
        for doc, score in hits:
            meta = dict(doc.metadata or {})
            meta["retrieved_from"] = source_name
            collected.append((doc.page_content, meta, float(score)))

    if intent == "guideline":
        preferred_tables = {
            "dga_rules_df.csv",
            "dga_standard_df.csv",
            "bmi_standard_df.csv",
            "standard_df.csv",
            "serving_size_reference_df.csv",
        }
        prioritized = [item for item in collected if str(item[1].get("table", "")) in preferred_tables]
        if prioritized:
            collected = prioritized + collected

    collected = sorted(collected, key=lambda x: x[2], reverse=True)[:6]
    context = "\n\n".join([item[0] for item in collected])
    top_score = collected[0][2] if collected else 0.0

    if (not collected) or (not context.strip()) or (top_score < 0.25):
        return {
            "question": question,
            "intent": intent,
            "answer": _default_fallback(question),
            "sources": [item[1] for item in collected[:2]],
        }

    prompt_text = rag_prompt.format(question=question, context=context)
    llm_response = llm.invoke(prompt_text)
    raw_answer = llm_response.content if hasattr(llm_response, "content") else str(llm_response)
    final_answer = _clean_chat_answer(raw_answer, intent=intent, question=question)

    return {
        "question": question,
        "intent": intent,
        "answer": final_answer,
        "sources": [item[1] for item in collected],
    }

In [19]:
demo_questions = [
    "BMI ปกติคือเท่าไร",
    "โซเดียมต่อวันควรไม่เกินเท่าไร",
    "น้ำตาลต่อมื้อควรไม่เกินเท่าไร",
    "cream เสี่ยงไหม",
    "อาหารที่มีคอเลสเตอรอลสูงมีตัวอย่างอะไรบ้าง",
    "อยากกินครีมซีสอะดีไหม แล้วมีค่าอะไรเท่าบ้าง",
    "Which users have high blood sugar and what meal plan is recommended?",
    "อาหารหมามีประโยชน์ไหม",
]

for q in demo_questions:
    print("=" * 100)
    response = ask_welly_rag(q)
    print("Question:", response["question"])
    print("Answer:", response["answer"])
    print("Sources:", response["sources"][:2])


Question: BMI ปกติคือเท่าไร
Answer: BMI ปกติคือระหว่าง 18.5 ถึง 22.99 กิโลกรัมต่อตารางเมตรครับ
Sources: [{'table': 'bmi_standard_df.csv', 'row_index': 2, 'retrieved_from': 'knowledge'}, {'table': 'bmi_standard_df.csv', 'row_index': 2, 'retrieved_from': 'knowledge'}]
Question: โซเดียมต่อวันควรไม่เกินเท่าไร
Answer: โซเดียมต่อวันควรไม่เกิน 2,300 มิลลิกรัม แต่สำหรับบางคนอาจต้องจำกัดลงเหลือ 1,500 มิลลิกรัม หากคุณมีปัญหาสุขภาพหรือมีข้อจำกัดด้านโภชนาการ ควรปรึกษากับแพทย์หรือผู้เชี่ยวชาญด้านโภชนาการเพื่อคำแนะนำที่เหมาะสมกับคุณครับ
Sources: [{'table': 'food_dataset_with_risk.csv', 'row_index': 628, 'retrieved_from': 'knowledge'}, {'table': 'food_dataset_with_risk.csv', 'row_index': 1564, 'retrieved_from': 'knowledge'}]
Question: น้ำตาลต่อมื้อควรไม่เกินเท่าไร
Answer: น้ำตาลต่อมื้อควรไม่เกิน 10 กรัมครับ
Sources: [{'table': 'dga_rules_df.csv', 'retrieved_from': 'knowledge', 'method': 'tabular_sugar_lookup'}, {'table': 'dga_standard_df.csv', 'retrieved_from': 'knowledge', 'method': 'tabular_sugar_l

## 13. Optional Utility: Food Name Search

ช่วยหาชื่ออาหารที่ใกล้เคียงกรณีพิมพ์ไม่ตรงเป๊ะ


In [20]:
def suggest_food_names(user_text, food_df):
    if "food_name" not in food_df.columns:
        return []

    food_names = food_df["food_name"].dropna().astype(str).unique().tolist()
    q = user_text.strip()

    fuzzy = process.extract(q, food_names, limit=5)
    close = get_close_matches(q, food_names, n=5, cutoff=0.4)

    result = []
    for item in fuzzy:
        result.append(item[0])
    for item in close:
        if item not in result:
            result.append(item)

    return result[:5]

if "food_dataset_with_risk.csv" in loaded_tables:
    food_df = loaded_tables["food_dataset_with_risk.csv"]
    print(suggest_food_names("chese", food_df))


['cheese spread', 'cheese croissant', 'cheese tortellini', 'cheese lasagna', 'cheese soup']


## 14. Build RAG Index for Recipes + Calories (Fast Mode)

In [21]:
recipes_path = Path("../data/13k-recipes.csv")
calories_path = Path("../data/calories.csv")

# Fast-mode knobs: tune these first when indexing is slow.
RECIPES_MAX_ROWS = 3000
RECIPE_INSTR_MAX_CHARS = 350

recipes_usecols = ["Title", "Cleaned_Ingredients", "Instructions"]
calories_usecols = ["FoodCategory", "FoodItem", "Cals_per100grams"]

recipes_df = pd.read_csv(recipes_path, usecols=recipes_usecols, nrows=RECIPES_MAX_ROWS)
calories_df = pd.read_csv(calories_path, usecols=calories_usecols)

recipes_df = recipes_df.fillna("")
calories_df = calories_df.fillna("")

recipe_docs = []
for idx, row in recipes_df.iterrows():
    title = str(row.get("Title", "")).strip()
    ingredients = str(row.get("Cleaned_Ingredients", "")).strip()
    instructions = str(row.get("Instructions", "")).strip()[:RECIPE_INSTR_MAX_CHARS]

    text = (
        f"Recipe. Title: {title}. "
        f"Ingredients: {ingredients}. "
        f"How to cook (summary): {instructions}."
    )

    recipe_docs.append(
        Document(
            page_content=text,
            metadata={
                "source": "13k-recipes.csv",
                "doc_type": "recipe",
                "row_index": int(idx),
                "title": title,
            },
        )
    )

calorie_docs = []
for idx, row in calories_df.iterrows():
    item = str(row.get("FoodItem", "")).strip()
    category = str(row.get("FoodCategory", "")).strip()
    cals = str(row.get("Cals_per100grams", "")).strip()

    text = (
        f"Calories reference. Food item: {item}. "
        f"Category: {category}. "
        f"Energy per 100 grams: {cals}."
    )

    calorie_docs.append(
        Document(
            page_content=text,
            metadata={
                "source": "calories.csv",
                "doc_type": "calorie",
                "row_index": int(idx),
                "food_item": item,
            },
        )
    )

recipe_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
recipe_split_docs = recipe_splitter.split_documents(recipe_docs)

print("Recipe rows loaded:", len(recipes_df))
print("Recipe chunks:", len(recipe_split_docs))
print("Calories rows loaded:", len(calories_df))
print("Calories docs:", len(calorie_docs))


Recipe rows loaded: 3000
Recipe chunks: 11608
Calories rows loaded: 2225
Calories docs: 2225


In [22]:
recipes_index_dir = Path(f"faiss_recipes_index_{model_tag}")
calories_index_dir = Path(f"faiss_calories_index_{model_tag}")

recipes_faiss = recipes_index_dir / "index.faiss"
recipes_pkl = recipes_index_dir / "index.pkl"

calories_faiss = calories_index_dir / "index.faiss"
calories_pkl = calories_index_dir / "index.pkl"

RECIPE_BUILD_BATCH_SIZE = 300
CALORIE_BUILD_BATCH_SIZE = 300


def build_faiss_in_batches(docs, embeddings, index_dir: Path, batch_size: int, label: str):
    total = len(docs)
    if total == 0:
        raise ValueError(f"No {label} docs available for indexing")

    print(f"Building {label} FAISS in batches. Total docs/chunks: {total}")

    first_end = min(batch_size, total)
    vs = FAISS.from_documents(docs[:first_end], embeddings)
    print(f"[{label}] Indexed {first_end}/{total}")

    for start in range(first_end, total, batch_size):
        end = min(start + batch_size, total)
        batch_docs = docs[start:end]

        batch_vs = FAISS.from_documents(batch_docs, embeddings)
        vs.merge_from(batch_vs)

        # Periodic checkpoint saves reduce rework if interrupted.
        if ((end // batch_size) % 2 == 0) or (end == total):
            index_dir.mkdir(parents=True, exist_ok=True)
            vs.save_local(str(index_dir))

        print(f"[{label}] Indexed {end}/{total}")

    index_dir.mkdir(parents=True, exist_ok=True)
    vs.save_local(str(index_dir))
    print(f"Built and saved {label} FAISS index")
    return vs


if recipes_faiss.exists() and recipes_pkl.exists():
    recipes_vectorstore = FAISS.load_local(
        str(recipes_index_dir),
        embeddings,
        allow_dangerous_deserialization=True,
    )
    print("Loaded existing recipes FAISS index")
else:
    recipes_vectorstore = build_faiss_in_batches(
        docs=recipe_split_docs,
        embeddings=embeddings,
        index_dir=recipes_index_dir,
        batch_size=RECIPE_BUILD_BATCH_SIZE,
        label="recipes",
    )

if calories_faiss.exists() and calories_pkl.exists():
    calories_vectorstore = FAISS.load_local(
        str(calories_index_dir),
        embeddings,
        allow_dangerous_deserialization=True,
    )
    print("Loaded existing calories FAISS index")
else:
    calories_vectorstore = build_faiss_in_batches(
        docs=calorie_docs,
        embeddings=embeddings,
        index_dir=calories_index_dir,
        batch_size=CALORIE_BUILD_BATCH_SIZE,
        label="calories",
    )

recipes_retriever = recipes_vectorstore.as_retriever(search_kwargs={"k": 4})
calories_retriever = calories_vectorstore.as_retriever(search_kwargs={"k": 4})

print("Recipes retriever ready")
print("Calories retriever ready")


Loaded existing recipes FAISS index
Loaded existing calories FAISS index
Recipes retriever ready
Calories retriever ready


In [23]:
demo_questions = [
    "เนื้อหมาอร่อยมั้ย",
]

for q in demo_questions:
    print("=" * 100)
    response = ask_welly_rag(q)
    print("Question:", response["question"])
    print("Answer:", response["answer"])
    print("Sources:", response["sources"][:2])


Question: เนื้อหมาอร่อยมั้ย
Answer: ตอนนี้ผมยังไม่มีข้อมูลที่เพียงพอเกี่ยวกับ 'เนื้อหมาอร่อยมั้ย'
หากคุณต้องการ ผมช่วยแนะนำคำถามด้านโภชนาการหรือสูตรอาหารที่ใกล้เคียงให้ได้ครับ
Sources: [{'table': 'food_dataset_with_risk.csv', 'row_index': 283, 'retrieved_from': 'knowledge'}, {'table': 'food_dataset_with_risk.csv', 'row_index': 894, 'retrieved_from': 'knowledge'}]


In [24]:
demo_questions = [
    "BMI ปกติคือเท่าไร",
]

for q in demo_questions:
    print("=" * 100)
    response = ask_welly_rag(q)
    print("Question:", response["question"])
    print("Intent:", response.get("intent"))
    print("Answer:", response["answer"])
    print("Sources:", response["sources"][:3])


Question: BMI ปกติคือเท่าไร
Intent: guideline
Answer: BMI ปกติคือระหว่าง 18.5 ถึง 22.99 กิโลกรัมต่อตารางเมตรครับ
Sources: [{'table': 'bmi_standard_df.csv', 'row_index': 2, 'retrieved_from': 'knowledge'}, {'table': 'bmi_standard_df.csv', 'row_index': 2, 'retrieved_from': 'knowledge'}, {'table': 'bmi_standard_df.csv', 'row_index': 4, 'retrieved_from': 'knowledge'}]


In [25]:
demo_questions = [
    "อยากได้เมนูที่ทำจาก cheese แต่ไม่เสียสุขภาพ มีอะไรบ้าง",
]

for q in demo_questions:
    print("=" * 100)
    response = ask_welly_rag(q)
    print("Question:", response["question"])
    print("Intent:", response.get("intent"))
    print("Answer:", response["answer"])
    print("Sources:", response["sources"][:3])


Question: อยากได้เมนูที่ทำจาก cheese แต่ไม่เสียสุขภาพ มีอะไรบ้าง
Intent: recipe
Answer: มีเมนูหลายอย่างที่ทำจาก cheese แต่ไม่เสียสุขภาพ นี่คือบางตัวอย่าง:

1. พิซซ่าแบบไม่เสียสุขภาพ โดยใช้ mozzarella cheese fat free, tomato sauce, และผักสด
2. ซาลาเปาแบบไม่เสียสุขภาพ โดยใช้ swiss cheese, ผักสด, และน้ำมะนาว
3. ซอสซอสแบบไม่เสียสุขภาพ โดยใช้ mozzarella cheese, น้ำมะนาว, และน้ำหอม
4. พายแบบไม่เสียสุขภาพ โดยใช้ cheshire cheese, ผักสด, และน้ำมะนาว

เมนูเหล่านี้สามารถทำได้โดยใช้ cheese ที่มีไขมันต่ำและไม่เกินเกณฑ์เบื้องต้น นอกจากนี้ยังสามารถเพิ่มผักสดและน้ำมะนาวเพื่อเพิ่มรสชาติและประโยชน์ต่อสุขภาพ
Sources: [{'table': 'food_dataset_with_risk.csv', 'row_index': 862, 'retrieved_from': 'knowledge'}, {'table': 'food_dataset_with_risk.csv', 'row_index': 37, 'retrieved_from': 'knowledge'}, {'table': 'food_dataset_with_risk.csv', 'row_index': 12, 'retrieved_from': 'knowledge'}]


In [26]:
demo_questions = [
    "แคลอรีหนึ่งวันควรกินเท่าไหร่",
]

for q in demo_questions:
    print("=" * 100)
    response = ask_welly_rag(q)
    print("Question:", response["question"])
    print("Intent:", response.get("intent"))
    print("Answer:", response["answer"])
    print("Sources:", response["sources"][:3])


Question: แคลอรีหนึ่งวันควรกินเท่าไหร่
Intent: calorie
Answer: ตอนนี้ผมยังไม่มีข้อมูลที่เพียงพอเกี่ยวกับ 'แคลอรีหนึ่งวันควรกินเท่าไหร่'
หากคุณต้องการ ผมช่วยแนะนำคำถามด้านโภชนาการหรือสูตรอาหารที่ใกล้เคียงให้ได้ครับ
Sources: [{'table': 'food_dataset_with_risk.csv', 'row_index': 76, 'retrieved_from': 'knowledge'}, {'table': 'standard_df.csv', 'row_index': 0, 'retrieved_from': 'knowledge'}, {'table': 'food_dataset_with_risk.csv', 'row_index': 1514, 'retrieved_from': 'knowledge'}]


## 14. Summary

Notebook นี้ถูกปรับให้เป็น RAG แบบ LLM-driven ตามลำดับงานมาตรฐาน:
- โหลดข้อมูลจากตารางในโปรเจกต์แล้วแปลงเป็น documents
- split เป็น chunks
- สร้าง embeddings และเก็บใน FAISS vector store
- รับ query แล้ว embed/query เข้า vector store เพื่อหา top-k context
- สร้าง contextual prompt (query + retrieved context)
- ให้ LLM สร้างคำตอบสุดท้ายจาก context

ดังนั้น answer path หลักตอนนี้คือ Query -> Retrieval -> Prompt Construction -> LLM -> Generated Response ตามภาพที่ต้องการ

In [27]:
quick_test_questions = [
    "อาหารที่มีคอเลสเตอรอลสูงมีตัวอย่างอะไรบ้าง",
    "น้ำตาลต่อวันควรไม่เกินเท่าไร",
]

for q in quick_test_questions:
    print("=" * 80)
    res = ask_welly_rag(q)
    print("Question:", res["question"])
    print("Intent:", res.get("intent"))
    print("Answer:")
    print(res["answer"])
    print("Sources:", res["sources"][:2])

Question: อาหารที่มีคอเลสเตอรอลสูงมีตัวอย่างอะไรบ้าง
Intent: cholesterol_examples
Answer:
ตัวอย่างอาหารที่มีคอเลสเตอรอลสูง (จาก dataset):
- veal brain cooked: 10509.0
- pork brain cooked: 9748.6
- lamb brain cooked: 7089.2
- beef brain cooked: 7002.5
- pork arm picnic cooked: 1936.7
Sources: [{'table': 'food_dataset_with_risk.csv', 'retrieved_from': 'knowledge', 'method': 'tabular_top_cholesterol'}]
Question: น้ำตาลต่อวันควรไม่เกินเท่าไร
Intent: guideline
Answer:
น้ำตาลที่ควรไม่เกิน 10 กรัมต่ออาหารแต่ละมื้อครับ
Sources: [{'table': 'dga_rules_df.csv', 'retrieved_from': 'knowledge', 'method': 'tabular_sugar_lookup'}, {'table': 'dga_standard_df.csv', 'retrieved_from': 'knowledge', 'method': 'tabular_sugar_lookup'}]
